# Analyse du dédoublonnage

## Fonction d'analyse
Calcul et restitution du nombre de ligne en défut d'intégrité

In [1]:
from datetime import datetime
import json
from tab_dataset import Cdataset
import pandas as pd
import ntv_pandas as npd
import pathlib

def analyse_integrite(data, schema, affiche=True, indic=True):
    '''analyse les relations du DataFrame 'data' définies dans le schéma 'schema'.
    Le nombre de lignes en erreur par relation (dict) est retourné et optionnellement affiché (paramètre 'affiche=True') . 
    Les lignes en erreur sont optionnellement ajoutées (paramètre 'indic=True') à 'data' sous forme de champs booléens par relation.
    '''
    dic_errors = Cdataset(data).check_relationship(schema)
    dic_count = {name: len(errors) for name, errors in dic_errors.items()}
    if affiche:
        for name, total in dic_count.items():
            print('{:<50} {:>5}'.format(name, total))
    if indic:
        data['ok'] = True
        for name, errors in dic_errors.items():
            data[name] = True
            data.loc[errors, name] = False
            data['ok'] = data['ok'] & data[name] 
    return dic_count

## Schéma de données
Le schéma de données restreint à la propriété 'relationship' et construit à partir du modèle de données est le suivants :

In [12]:
# complément à inclure dans le schéma de données
schema = {
    'relationships': [
         # relation unicité des pdl
         {"fields": ["id_pdc_itinerance", "index"],                    "link" : "coupled" },   
         # relations inter entités
         {"fields": ["id_station_itinerance", "contact_operateur"],    "link" : "derived" },
         {"fields": ["id_station_itinerance", "nom_enseigne"],         "link" : "derived" },
         {"fields": ["id_station_itinerance", "coordonneesXY"],        "link" : "derived" },
         {"fields": ["id_pdc_itinerance", "id_station_itinerance"],    "link" : "derived" },
         # relations intra entité - station
         {"fields": ["id_station_itinerance", "nom_station"],          "link" : "derived" },
         {"fields": ["id_station_itinerance", "implantation_station"], "link" : "derived" },
         #{"fields": ["id_station_itinerance", "date_maj"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "nbre_pdc"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "condition_acces"],      "link" : "derived" },
         {"fields": ["id_station_itinerance", "horaires"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "station_deux_roues"],   "link" : "derived" },
         # relations intra entité - localisation
         {"fields": ["coordonneesXY", "adresse_station"],              "link" : "derived" }
    ]
}

## Initialisation des données
Fichier pandas

In [17]:
file_irve_brut = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260301.csv'
file_irve = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260401.csv'

irve_brut = pd.read_csv(file_irve_brut, sep=',', low_memory=False, dtype='object').reset_index()
irve_brut['last_modified'] = irve_brut['datagouv_last_modified']

irve = pd.read_csv(file_irve, sep=',', low_memory=False, dtype='object').reset_index()
irve['last_modified'] = irve['datagouv_last_modified']

print('nombre de lignes : ', len(irve_brut), len(irve))

nombre de lignes :  379946 148286


## Analyse d'intégrité
Qualification du dédoublonnage

In [22]:
resultat = analyse_integrite(irve, schema)

print("\nnombre d'enregistrements sans erreurs : ", sum(irve['ok']))
print("nombre d'enregistrements avec au moins une erreur : ", len(irve) - sum(irve['ok']))
print("dont doublons : ", resultat['index - id_pdc_itinerance'])
print("\ntaux d'erreur : ", round(len(irve_ko) / len(irve) * 100), ' %')

index - id_pdc_itinerance                            816
contact_operateur - id_station_itinerance           1777
nom_enseigne - id_station_itinerance                4064
coordonneesXY - id_station_itinerance               5402
id_station_itinerance - id_pdc_itinerance             28
nom_station - id_station_itinerance                 2436
implantation_station - id_station_itinerance        2408
nbre_pdc - id_station_itinerance                    5166
condition_acces - id_station_itinerance              280
horaires - id_station_itinerance                    1079
station_deux_roues - id_station_itinerance            73
adresse_station - coordonneesXY                     8224

nombre d'enregistrements sans erreurs :  132857
nombre d'enregistrements avec au moins une erreur :  15429
nombre d'enregistrements avec au moins une erreur :  15429
dont doublons :  816

taux d'erreur :  10  %
